# ARC + MolFormer — Clean Pipeline (v14)
## Changes from v13
| # | Change |
|---|--------|
| 1 | ATC dataset removed — replaced by BBBP + ClinTox + HIV |
| 2 | Three binary datasets combined into 6-class CIL sequence (3 tasks × 2 classes) |
| 3 | BBBP (Task 0), ClinTox (Task 1), HIV (Task 2) |
| 4 | DrugBank pipeline unchanged |
| 5 | All ATC variable names replaced with `mds_` (multi-dataset) |
| 6 | HIV class imbalance handled via weighted CE loss (same as v13 mechanism) |


## Step 1 — Install & Imports

In [1]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

packages = [
    'torch>=2.0.0',
    'transformers>=4.35.0',
    'scikit-learn>=1.3.0',
    'numpy>=1.24.0',
    'matplotlib>=3.7.0',
    'seaborn>=0.12.0',
    'pandas>=2.0.0',
    'rdkit',
    'tqdm',
]
for p in packages:
    install(p)
print('All packages installed')

All packages installed


In [1]:
import os, json, copy, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import MolStandardize, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import roc_auc_score, f1_score

from transformers import AutoTokenizer, AutoModel

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print('All imports OK')

Device: cuda
All imports OK


## Step 2 — Config

In [6]:
CFG = {
    'bbbp_path'         : 'bbbp_singlelabel.csv',
    'clintox_path'      : 'clintox (1).csv',
    'hiv_path'          : 'HIV_singlelabel.csv',
    'drugbank_path'     : 'df_drugbank_smiles.csv',
    'output_dir'        : 'arc_output',
    'min_ha'            : 5,
    'max_ha'            : 100,
    'min_mw'            : 100,
    'max_mw'            : 1500,
    'train_ratio'       : 0.70,
    'val_ratio'         : 0.15,
    'test_ratio'        : 0.15,
    'classes_per_task'  : 2,
    'molformer_name'    : 'ibm/MoLFormer-XL-both-10pct',
    'molformer_batch'   : 32,
    'max_smiles_len'    : 202,
    'clf_epochs'        : 100,
    'clf_lr'            : 1e-3,
    'clf_batch'         : 32,
    'arc_epsilon'       : 0.70,   # Assumption 1 threshold (paper Fig 5)
    'arc_theta'         : 0.05,   # Assumption 2 threshold (paper Fig 5)
    'arc_temp'          : 2.0,    # TSS temperature T (paper Def 1)
    'arc_lr'            : 1e-4,   # Lowered from 1e-5 to reduce retention drift
    'seed'              : 42,
    'es_patience'       : 10,
    'es_min_delta'      : 1e-4,
    'min_class_samples' : 30,
    'ewc_lambda'        : 1000.0,  # EWC penalty strength
}
os.makedirs(CFG['output_dir'], exist_ok=True)
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
print('Config ready')
print(f"  arc_epsilon={CFG['arc_epsilon']}  arc_theta={CFG['arc_theta']}  arc_temp={CFG['arc_temp']}  arc_lr={CFG['arc_lr']}")
print(f"  ewc_lambda={CFG['ewc_lambda']}")


Config ready
  arc_epsilon=0.7  arc_theta=0.05  arc_temp=2.0  arc_lr=0.0001
  ewc_lambda=1000.0


## Step 3 — Multi-Dataset Pipeline (BBBP → ClinTox → HIV)
Each dataset contributes 2 classes → 3 CIL tasks, 6 classes total.
- Task 0: BBBP       (label_id 0=BBBP-neg,      1=BBBP-pos)
- Task 1: ClinTox    (label_id 2=ClinTox-safe,   3=ClinTox-toxic)
- Task 2: HIV        (label_id 4=HIV-inactive,   5=HIV-active)


In [7]:
def clean_smiles(smiles, cfg):
    if not isinstance(smiles, str) or smiles.strip() == '':
        return None, 'missing_or_empty'
    mol = Chem.MolFromSmiles(smiles.strip())
    if mol is None:
        return None, 'invalid_smiles'
    try:
        mol = MolStandardize.rdMolStandardize.LargestFragmentChooser().choose(mol)
    except Exception:
        pass
    try:
        mol = MolStandardize.rdMolStandardize.Uncharger().uncharge(mol)
        mol = MolStandardize.rdMolStandardize.TautomerEnumerator().Canonicalize(mol)
    except Exception:
        pass
    try:
        ha = mol.GetNumHeavyAtoms()
        mw = rdMolDescriptors.CalcExactMolWt(mol)
        if not (cfg['min_ha'] <= ha <= cfg['max_ha'] and cfg['min_mw'] <= mw <= cfg['max_mw']):
            return None, 'filtered_out'
    except Exception:
        return None, 'filter_error'
    canon = Chem.MolToSmiles(mol, canonical=True)
    return (canon, 'ok') if canon else (None, 'canon_failed')


def get_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return smiles
        sc = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(sc, canonical=True)
    except Exception:
        return smiles

print('SMILES helpers defined')

SMILES helpers defined


In [8]:
def stratified_scaffold_split(df, label_col, train_r, val_r, test_r, seed=42):
    assert abs(train_r + val_r + test_r - 1.0) < 1e-6
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for cls in sorted(df[label_col].unique()):
        cls_df = df[df[label_col] == cls]
        sc2idx = defaultdict(list)
        for idx, sc in zip(cls_df.index, cls_df['scaffold']):
            sc2idx[sc].append(idx)
        groups = list(sc2idx.values())
        rng.shuffle(groups)
        n    = len(cls_df)
        n_tr = max(1, int(n * train_r))
        n_vl = max(1, int(n * val_r))
        cls_tr, cls_vl, cls_ts = [], [], []
        for g in groups:
            if len(cls_tr) < n_tr:
                cls_tr.extend(g)
            elif len(cls_vl) < n_vl:
                cls_vl.extend(g)
            else:
                cls_ts.extend(g)
        if len(cls_vl) == 0 and len(cls_tr) > 1:
            cls_vl.append(cls_tr.pop())
        if len(cls_ts) == 0 and len(cls_tr) > 1:
            cls_ts.append(cls_tr.pop())
        train_idx.extend(cls_tr)
        val_idx.extend(cls_vl)
        test_idx.extend(cls_ts)
    tr = df.loc[train_idx].copy()
    vl = df.loc[val_idx].copy()
    ts = df.loc[test_idx].copy()
    return tr, vl, ts


def verify_split(train_df, val_df, test_df, label_col='label_id'):
    for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        dist = df[label_col].value_counts().sort_index().to_dict()
        print(f'  {name:5s}: {len(df):4d} samples | classes: {dist}')
    tsc = set(train_df['scaffold']); vsc = set(val_df['scaffold']); esc = set(test_df['scaffold'])
    print(f'  Scaffold leakage → Train∩Val={len(tsc&vsc)} | Train∩Test={len(tsc&esc)} | Val∩Test={len(vsc&esc)}')

print('Stratified scaffold split defined')

Stratified scaffold split defined


In [9]:
# ── Load and clean each dataset ──────────────────────────────────────────────
def load_binary_dataset(path, smiles_col, label_col, label_offset, cfg,
                         neg_name, pos_name):
    """
    Load a binary (0/1) molecular dataset, clean SMILES,
    remap labels to global CIL label space via label_offset.
    Returns cleaned dataframe with columns: canon_smiles, label_id, scaffold.
    """
    raw = pd.read_csv(path)
    print(f'  Raw: {raw.shape}  |  label counts: {raw[label_col].value_counts().to_dict()}')

    res = raw[smiles_col].apply(lambda s: clean_smiles(s, cfg))
    raw['canon_smiles']  = res.apply(lambda x: x[0])
    raw['clean_status']  = res.apply(lambda x: x[1])
    df = raw[raw['clean_status'] == 'ok'].drop_duplicates('canon_smiles').copy()
    print(f'  After cleaning: {df.shape}')

    # Remap 0/1 → global offset
    df['label_id'] = df[label_col].astype(int) + label_offset
    df['scaffold'] = df['canon_smiles'].apply(get_scaffold)
    return df


print('=== Loading BBBP ===')
bbbp_df = load_binary_dataset(
    CFG['bbbp_path'], smiles_col='smiles', label_col='label',
    label_offset=0, cfg=CFG,
    neg_name='BBBP-neg', pos_name='BBBP-pos'
)

print('\n=== Loading ClinTox ===')
clintox_df = load_binary_dataset(
    CFG['clintox_path'], smiles_col='SMILES', label_col='Label',
    label_offset=2, cfg=CFG,
    neg_name='ClinTox-safe', pos_name='ClinTox-toxic'
)

print('\n=== Loading HIV ===')
hiv_df = load_binary_dataset(
    CFG['hiv_path'], smiles_col='smiles', label_col='label',
    label_offset=4, cfg=CFG,
    neg_name='HIV-inactive', pos_name='HIV-active'
)

# Global label map for all 6 classes
mds_label_map = {
    0: 'BBBP-neg',      1: 'BBBP-pos',
    2: 'ClinTox-safe',  3: 'ClinTox-toxic',
    4: 'HIV-inactive',  5: 'HIV-active',
}
print('\nGlobal label map:', mds_label_map)


=== Loading BBBP ===
  Raw: (2050, 2)  |  label counts: {1: 1567, 0: 483}
  After cleaning: (1925, 4)

=== Loading ClinTox ===
  Raw: (1478, 3)  |  label counts: {0: 1366, 1: 112}
  After cleaning: (1363, 5)

=== Loading HIV ===
  Raw: (41127, 2)  |  label counts: {0: 39684, 1: 1443}
  After cleaning: (40789, 4)

Global label map: {0: 'BBBP-neg', 1: 'BBBP-pos', 2: 'ClinTox-safe', 3: 'ClinTox-toxic', 4: 'HIV-inactive', 5: 'HIV-active'}


In [10]:
# ── Stratified scaffold split per dataset, then combine ──────────────────────
def split_dataset(df, name, cfg):
    tr, vl, ts = stratified_scaffold_split(
        df, 'label_id', cfg['train_ratio'], cfg['val_ratio'], cfg['test_ratio'], cfg['seed']
    )
    print(f'\n{name}: {df.shape}  train={len(tr)} val={len(vl)} test={len(ts)}')
    verify_split(tr, vl, ts)
    return tr, vl, ts


bbbp_tr,    bbbp_vl,    bbbp_ts    = split_dataset(bbbp_df,    'BBBP',    CFG)
clintox_tr, clintox_vl, clintox_ts = split_dataset(clintox_df, 'ClinTox', CFG)
hiv_tr,     hiv_vl,     hiv_ts     = split_dataset(hiv_df,     'HIV',     CFG)

# Combined splits (for unified feature extraction and CIL pipeline)
mds_df    = pd.concat([bbbp_df,    clintox_df,  hiv_df],    ignore_index=True)
mds_train = pd.concat([bbbp_tr,    clintox_tr,  hiv_tr],    ignore_index=True)
mds_val   = pd.concat([bbbp_vl,    clintox_vl,  hiv_vl],    ignore_index=True)
mds_test  = pd.concat([bbbp_ts,    clintox_ts,  hiv_ts],    ignore_index=True)

print(f'\nCombined MDS: total={len(mds_df)}  train={len(mds_train)} val={len(mds_val)} test={len(mds_test)}')
print('Class distribution in combined train:')
print(mds_train['label_id'].value_counts().sort_index())



BBBP: (1925, 6)  train=1346 val=289 test=290
  Train: 1346 samples | classes: {0: 324, 1: 1022}
  Val  :  289 samples | classes: {0: 69, 1: 220}
  Test :  290 samples | classes: {0: 71, 1: 219}
  Scaffold leakage → Train∩Val=6 | Train∩Test=9 | Val∩Test=2

ClinTox: (1363, 7)  train=1066 val=204 test=93
  Train: 1066 samples | classes: {2: 1014, 3: 52}
  Val  :  204 samples | classes: {2: 193, 3: 11}
  Test :   93 samples | classes: {2: 81, 3: 12}
  Scaffold leakage → Train∩Val=4 | Train∩Test=2 | Val∩Test=1

HIV: (40789, 6)  train=28553 val=6124 test=6112
  Train: 28553 samples | classes: {4: 27576, 5: 977}
  Val  : 6124 samples | classes: {4: 5908, 5: 216}
  Test : 6112 samples | classes: {4: 5909, 5: 203}
  Scaffold leakage → Train∩Val=68 | Train∩Test=86 | Val∩Test=17

Combined MDS: total=44077  train=30965 val=6617 test=6495
Class distribution in combined train:
label_id
0      324
1     1022
2     1014
3       52
4    27576
5      977
Name: count, dtype: int64


In [11]:
def make_cil_tasks(train_df, val_df, test_df, cpt, label_map):
    all_ids = sorted(train_df['label_id'].unique())
    tasks = []
    for t in range(int(np.ceil(len(all_ids) / cpt))):
        new_cls  = all_ids[t*cpt : (t+1)*cpt]
        seen_cls = all_ids[: (t+1)*cpt]
        tasks.append({
            'task_id'    : t,
            'new_classes': new_cls,
            'all_classes': list(seen_cls),
            'n_classes'  : len(seen_cls),
            'class_names': {lid: label_map[lid] for lid in seen_cls},
        })
    return tasks


mds_tasks = make_cil_tasks(mds_train, mds_val, mds_test, CFG['classes_per_task'], mds_label_map)
print(f'Multi-Dataset CIL tasks: {len(mds_tasks)}')
for t in mds_tasks:
    print(f"  Task {t['task_id']}: new={[t['class_names'][c] for c in t['new_classes']]}  all_classes={t['all_classes']}")


Multi-Dataset CIL tasks: 3
  Task 0: new=['BBBP-neg', 'BBBP-pos']  all_classes=[0, 1]
  Task 1: new=['ClinTox-safe', 'ClinTox-toxic']  all_classes=[0, 1, 2, 3]
  Task 2: new=['HIV-inactive', 'HIV-active']  all_classes=[0, 1, 2, 3, 4, 5]


## Step 4 — DrugBank Data Pipeline

In [12]:
db_raw = pd.read_csv(CFG['drugbank_path'])
print(f'DrugBank raw: {db_raw.shape}')

smiles_col_db = 'smiles' if 'smiles' in db_raw.columns else 'SMILES'
res_db = db_raw[smiles_col_db].apply(lambda s: clean_smiles(s, CFG))
db_raw['canon_smiles'] = res_db.apply(lambda x: x[0])
db_raw['clean_status'] = res_db.apply(lambda x: x[1])

db_df = db_raw[db_raw['clean_status'] == 'ok'].drop_duplicates('canon_smiles').copy()
print(f'After cleaning: {db_df.shape}')

IGNORE_COLS = {
    'Unnamed: 0', 'drugbank_id', 'name', 'cas', 'smiles', 'SMILES',
    'canon_smiles', 'clean_status', 'logP ALOGPS', 'logP ChemAxon',
    'solubility ALOGPS', 'pKa (strongest acidic)', 'pKa (strongest basic)', 'F'
}
DB_LABEL_COLS = [c for c in db_df.columns if c not in IGNORE_COLS]

def assign_db_label(row):
    vals = row[DB_LABEL_COLS].fillna(0)
    if vals.sum() == 0:
        return 'none_detected'
    return vals.idxmax()

db_df['primary_label'] = db_df.apply(assign_db_label, axis=1)

MIN_CLASS_SAMPLES = CFG['min_class_samples']
label_counts  = db_df['primary_label'].value_counts()
valid_labels  = label_counts[label_counts >= MIN_CLASS_SAMPLES].index.tolist()
db_df         = db_df[db_df['primary_label'].isin(valid_labels)].copy()

le_db = LabelEncoder()
db_df['label_id'] = le_db.fit_transform(db_df['primary_label'])
db_label_map = {int(e): c for e, c in enumerate(le_db.classes_)}

db_df['scaffold'] = db_df['canon_smiles'].apply(get_scaffold)
db_train, db_val, db_test = stratified_scaffold_split(
    db_df, 'label_id', CFG['train_ratio'], CFG['val_ratio'], CFG['test_ratio'], CFG['seed']
)
print(f'DrugBank: {db_df.shape}  →  train={len(db_train)} val={len(db_val)} test={len(db_test)}')
verify_split(db_train, db_val, db_test)

db_tasks = make_cil_tasks(db_train, db_val, db_test, CFG['classes_per_task'], db_label_map)
print(f'\nDrugBank CIL tasks: {len(db_tasks)}')
for t in db_tasks:
    print(f"  Task {t['task_id']}: new={[t['class_names'][c] for c in t['new_classes']]}")

DrugBank raw: (5653, 16)
After cleaning: (5361, 18)
DrugBank: (5339, 21)  →  train=3746 val=800 test=793
  Train: 3746 samples | classes: {0: 2673, 1: 37, 2: 894, 3: 142}
  Val  :  800 samples | classes: {0: 572, 1: 6, 2: 190, 3: 32}
  Test :  793 samples | classes: {0: 574, 1: 1, 2: 189, 3: 29}
  Scaffold leakage → Train∩Val=25 | Train∩Test=27 | Val∩Test=12

DrugBank CIL tasks: 2
  Task 0: new=['carbonyl', 'nitro']
  Task 1: new=['none_detected', 'sulfonyl']


## Step 5 — MolFormer Feature Extraction

In [13]:
print('Loading MolFormer tokenizer & model...')
tokenizer = AutoTokenizer.from_pretrained(CFG['molformer_name'], trust_remote_code=True)
molformer = AutoModel.from_pretrained(
    CFG['molformer_name'], trust_remote_code=True, deterministic_eval=True
)
molformer.eval().to(DEVICE)
FEAT_DIM = molformer.config.hidden_size

for p in molformer.parameters():
    p.requires_grad = False

print(f'MolFormer loaded  |  hidden_dim={FEAT_DIM}')
print('Backbone frozen (ARC paper: frozen feature extractor assumption)')

Loading MolFormer tokenizer & model...
MolFormer loaded  |  hidden_dim=768
Backbone frozen (ARC paper: frozen feature extractor assumption)


In [14]:
@torch.no_grad()
def extract_molformer_features(smiles_list, batch_size=32):
    all_emb = []
    for i in tqdm(range(0, len(smiles_list), batch_size), desc='MolFormer encode'):
        batch = smiles_list[i: i + batch_size]
        enc   = tokenizer(
            batch, padding=True, truncation=True,
            max_length=CFG['max_smiles_len'], return_tensors='pt'
        ).to(DEVICE)
        out    = molformer(**enc)
        hidden = out.last_hidden_state
        mask   = enc['attention_mask']
        mask_f = mask.unsqueeze(-1).float()
        summed = (hidden * mask_f).sum(dim=1)
        counts = mask_f.sum(dim=1).clamp(min=1)
        pooled = (summed / counts).cpu().numpy()
        all_emb.append(pooled)
    return np.vstack(all_emb)

def get_feats(split_df, idx_to_feat):
    X = np.stack([idx_to_feat[i] for i in split_df.index])
    y = split_df['label_id'].values
    return X, y

print('Feature extractor ready')

Feature extractor ready


In [15]:
print('=== Extracting Multi-Dataset (BBBP+ClinTox+HIV) features ===')
mds_feats_all = extract_molformer_features(mds_df['canon_smiles'].tolist(), CFG['molformer_batch'])
idx_to_feat_mds = {idx: feat for idx, feat in zip(mds_df.index, mds_feats_all)}

mds_X_tr,  mds_y_tr  = get_feats(mds_train, idx_to_feat_mds)
mds_X_val, mds_y_val = get_feats(mds_val,   idx_to_feat_mds)
mds_X_te,  mds_y_te  = get_feats(mds_test,  idx_to_feat_mds)
print(f'MDS → train:{mds_X_tr.shape} | val:{mds_X_val.shape} | test:{mds_X_te.shape}')

print('\n=== Extracting DrugBank features ===')
db_feats_all = extract_molformer_features(db_df['canon_smiles'].tolist(), CFG['molformer_batch'])
idx_to_feat_db = {idx: feat for idx, feat in zip(db_df.index, db_feats_all)}

db_X_tr,  db_y_tr  = get_feats(db_train, idx_to_feat_db)
db_X_val, db_y_val = get_feats(db_val,   idx_to_feat_db)
db_X_te,  db_y_te  = get_feats(db_test,  idx_to_feat_db)
print(f'DrugBank → train:{db_X_tr.shape} | val:{db_X_val.shape} | test:{db_X_te.shape}')


=== Extracting Multi-Dataset (BBBP+ClinTox+HIV) features ===


MolFormer encode:   0%|          | 0/1378 [00:00<?, ?it/s]

MDS → train:(30965, 768) | val:(6617, 768) | test:(6495, 768)

=== Extracting DrugBank features ===


MolFormer encode:   0%|          | 0/167 [00:00<?, ?it/s]

DrugBank → train:(3746, 768) | val:(800, 768) | test:(793, 768)


## Step 6 — CIL Classifier + EWC

In [16]:
class MolDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class ExpandingLinearHead(nn.Module):
    """Single linear layer that grows as new tasks arrive."""
    def __init__(self, in_dim, n_init):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_init)

    def grow(self, n_new, device):
        old   = self.fc
        n_old = old.out_features
        new_fc = nn.Linear(old.in_features, n_old + n_new)
        with torch.no_grad():
            new_fc.weight[:n_old] = old.weight
            new_fc.bias[:n_old]   = old.bias
            nn.init.xavier_uniform_(new_fc.weight[n_old:])
            nn.init.zeros_(new_fc.bias[n_old:])
        self.fc = new_fc.to(device)

    def forward(self, x): return self.fc(x)

    @property
    def n_classes(self): return self.fc.out_features


class EWC:
    """
    Elastic Weight Consolidation (Kirkpatrick et al., 2017).
    Computes Fisher diagonal after each task; adds quadratic penalty
    during next task training to protect important weights.
    Training-only — has no effect during evaluation.
    """
    def __init__(self, model, dataloader, device, n_classes):
        self.params = {n: p.clone().detach()
                       for n, p in model.named_parameters() if p.requires_grad}
        self.fisher  = self._compute_fisher(model, dataloader, device, n_classes)

    def _compute_fisher(self, model, dataloader, device, n_classes):
        fisher = {n: torch.zeros_like(p)
                  for n, p in model.named_parameters() if p.requires_grad}
        model.eval()
        for xb, yb in dataloader:
            xb, yb = xb.to(device), yb.to(device)
            model.zero_grad()
            logits = model(xb)
            mask   = yb < n_classes
            if mask.sum() == 0:
                continue
            loss = F.cross_entropy(logits[mask], yb[mask])
            loss.backward()
            for n, p in model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    fisher[n] += p.grad.detach().pow(2)
        n_batches = max(len(dataloader), 1)
        for n in fisher:
            fisher[n] /= n_batches
        return fisher

    def penalty(self, model):
        loss = 0.0
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.fisher:
                f   = self.fisher[n]
                p0  = self.params[n]
                slices = tuple(slice(0, s) for s in f.shape)
                loss += (f * (p[slices] - p0).pow(2)).sum()
        return loss


print('ExpandingLinearHead + EWC defined')

ExpandingLinearHead + EWC defined


In [17]:
def compute_macro_f1(clf, X_v, y_v, device):
    if len(X_v) == 0:
        return float('nan')
    clf.eval()
    with torch.no_grad():
        xv   = torch.tensor(X_v, dtype=torch.float32).to(device)
        pred = clf(xv).argmax(1).cpu().numpy()
    present = np.unique(y_v)
    return float(f1_score(y_v, pred, labels=present, average='macro', zero_division=0))


def train_classifier_on_task(clf, X_tr, y_tr, X_val, y_val,
                              seen_cls, epochs, lr, batch_size, device,
                              es_patience=10, es_min_delta=1e-4,
                              ewc=None, ewc_lambda=400.0):
    """
    Memory-free training with optional EWC regularization.
    No replay buffer — pure CIL setting matching ARC paper.
    EWC penalty protects weights from previous tasks during training.
    """
    tr_mask  = np.isin(y_tr, seen_cls)
    X_t, y_t = X_tr[tr_mask], y_tr[tr_mask]
    val_mask  = np.isin(y_val, seen_cls)
    X_v, y_v  = X_val[val_mask], y_val[val_mask]

    n_total_cls  = clf.n_classes
    weight_tensor = torch.ones(n_total_cls, dtype=torch.float32).to(device)
    seen_counts   = np.array([max(1, (y_t == c).sum()) for c in seen_cls])
    seen_weights  = 1.0 / seen_counts.astype(float)
    seen_weights  = seen_weights / seen_weights.sum() * len(seen_cls)
    for idx, c in enumerate(seen_cls):
        weight_tensor[c] = float(seen_weights[idx])
        
    ds = MolDataset(X_t, y_t)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    opt     = torch.optim.Adam(clf.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)

    best_val_f1    = -1.0
    best_weights   = copy.deepcopy(clf.state_dict())
    patience_count = 0

    clf.train()
    for ep in range(epochs):
        ep_loss = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            task_loss = loss_fn(clf(xb), yb)
            ewc_loss  = ewc.penalty(clf) if ewc is not None else 0.0
            loss      = task_loss + ewc_lambda * ewc_loss
            loss.backward()
            opt.step()
            ep_loss += loss.item()

        val_f1 = compute_macro_f1(clf, X_v, y_v, device)
        if not np.isnan(val_f1) and val_f1 - best_val_f1 >= es_min_delta:
            best_val_f1    = val_f1
            best_weights   = copy.deepcopy(clf.state_dict())
            patience_count = 0
        else:
            patience_count += 1

        if (ep + 1) % 10 == 0:
            ewc_str = f'  ewc={float(ewc_loss):.5f}' if ewc is not None else ''
            print(f'  Epoch {ep+1:3d}/{epochs}  loss={ep_loss/max(len(dl),1):.4f}'
                  f'  val_f1={val_f1:.3f}{ewc_str}  patience={patience_count}/{es_patience}')

        if patience_count >= es_patience:
            print(f'  [Early Stop] epoch {ep+1}  best_val_f1={best_val_f1:.4f}')
            break
        clf.train()

    clf.load_state_dict(best_weights)
    clf.eval()
    return clf


print('train_classifier_on_task defined (memory-free + EWC + weighted CE + macro-F1 ES)')

train_classifier_on_task defined (memory-free + EWC + weighted CE + macro-F1 ES)


## Step 7 — Out-of-Task Detection (OTD)
Paper Algorithm 1 — two assumptions:
- **Assumption 1 (Retention):** pred ∈ past-task classes AND confidence c ≥ ε
- **Assumption 2 (Correction):** pred ∈ current-task classes AND w = c/ĉ < ϑ  
  where ĉ = max prob over past-task classes only

In [18]:
def compute_batch_theta(logits_all, task_id, n_classes_per_task, theta):
    """
    Adaptive batch-level theta: median of w=c/c_hat across the batch.
    Clips to [theta, 3.0] — prevents extreme values.
    """
    s             = n_classes_per_task
    past_boundary = s * task_id
    if past_boundary == 0:
        return theta
    with torch.no_grad():
        probs     = F.softmax(logits_all.float(), dim=-1)
        conf_all  = probs.max(dim=-1).values
        c_hat_all = probs[:, :past_boundary].max(dim=-1).values
        w_all     = conf_all / (c_hat_all + 1e-8)
    adaptive = float(w_all.median()) * 1.2
    return float(np.clip(adaptive, theta, 3.0))


def out_of_task_detection(logits, task_id, n_classes_per_task, epsilon, theta,
                           batch_theta=None):
    """
    OTD: Algorithm 1, ARC paper (Chen et al., ICLR 2025).

    Assumption 1 → retention:
      pred in past classes AND c >= epsilon
    Assumption 2 → correction:
      pred in current classes AND w = c/c_hat < theta
      (c_hat = max prob over past classes only — paper Eq. after Assumption 2)

    Returns: decisions list, probs tensor, pred_cls tensor
    """
    s             = n_classes_per_task
    past_boundary = s * task_id

    probs    = F.softmax(logits.float(), dim=-1)
    pred_cls = probs.argmax(dim=-1)
    conf_all = probs.max(dim=-1).values

    effective_theta = batch_theta if batch_theta is not None else theta

    decisions = []
    for i in range(logits.shape[0]):
        pred_i = pred_cls[i].item()
        c      = conf_all[i].item()

        if past_boundary == 0:
            decisions.append('current')
            continue

        if pred_i < past_boundary:
            # Assumption 1: past-task class predicted with high confidence → retention
            if c >= epsilon:
                decisions.append('retention')
            else:
                decisions.append('current')
        else:
            # Assumption 2: current-task class predicted, check w = c/c_hat
            c_hat = probs[i, :past_boundary].max().item()
            w     = c / (c_hat + 1e-8)
            if w < effective_theta:
                decisions.append('correction')
            else:
                decisions.append('current')

    return decisions, probs, pred_cls


print('OTD defined (paper Algorithm 1: Assumption 1 retention | Assumption 2 correction)')

OTD defined (paper Algorithm 1: Assumption 1 retention | Assumption 2 correction)


## Step 8 — Adaptive Retention (Paper Eq. 2 & 3)
L = L_CE(pseudo_label) + L_EM (entropy minimization)  
One SGD step per sample. Gradient clipped at 0.5 to prevent drift.

In [19]:
def adaptive_retention_step(clf, x_single, pseudo_label, lr, device):
    """
    Adaptive Retention — ARC paper Eq. 2 & 3.
    L_CE: cross-entropy with pseudo-label (OTD-assigned past class).
    L_EM: entropy minimization for prediction confidence.
    One SGD gradient update per sample (paper Section 3.4.1).
    Gradient clipped at max_norm=0.5 to prevent cumulative drift.
    """
    clf.train()
    opt = torch.optim.SGD(clf.parameters(), lr=lr)
    opt.zero_grad()

    x = x_single.detach().clone().to(device)

    logits  = clf(x)
    probs   = F.softmax(logits, dim=-1)
    label_t = torch.tensor([pseudo_label], dtype=torch.long).to(device)

    # Paper Eq. 2
    L_CE = F.cross_entropy(logits, label_t)
    L_EM = -(probs * probs.clamp(min=1e-8).log()).sum(dim=-1).mean()

    # Paper Eq. 3
    loss = L_CE + L_EM
    loss.backward()

    # Gradient clipping — prevents single bad pseudo-label from destroying weights
    torch.nn.utils.clip_grad_norm_(clf.parameters(), max_norm=0.5)

    opt.step()
    clf.eval()


print('Adaptive Retention defined (L_CE + L_EM, 1 SGD step, grad clip=0.5)')

Adaptive Retention defined (L_CE + L_EM, 1 SGD step, grad clip=0.5)


## Step 9 — Adaptive Correction: Task-based Softmax Score (TSS)
Paper Definition 1:  
S_i = max_{s(i-1)≤k<s·i} exp(z_k / T^(t-i)) / Σ_{j=0}^{s·i-1} exp(z_j / T^(t-i))  
Temperature T^(t-i) discounts older tasks less — corrects bias among past tasks.

In [20]:
def compute_tss(logits_single, task_id, n_classes_per_task, temperature):
    """
    Task-based Softmax Score (TSS) — ARC paper Definition 1.
    Picks the task Ti with highest score, returns its best class.
    Temperature T^(t-i) re-balances across past tasks so earlier
    tasks are not further suppressed by the accumulating denominator.
    """
    s = n_classes_per_task
    t = task_id + 1
    z = logits_single.float()

    best_score = -float('inf')
    best_task  = 0
    best_cls   = 0

    for i in range(1, t + 1):
        exponent   = t - i
        temp_i     = temperature ** exponent

        task_logits   = z[s * (i - 1) : s * i]
        scaled_task   = task_logits / temp_i
        numerator     = scaled_task.exp().max()

        prefix_logits = z[: s * i]
        denominator   = (prefix_logits / temp_i).exp().sum()

        score = (numerator / (denominator + 1e-12)).item()

        if score > best_score:
            best_score = score
            best_task  = i - 1
            best_cls   = s * (i - 1) + task_logits.argmax().item()

    return best_task, best_cls


print('TSS (Adaptive Correction) defined — paper Definition 1')

TSS (Adaptive Correction) defined — paper Definition 1


## Step 10 — Hyperparameter Sweep for ε and ϑ (Paper Fig 5)

In [21]:
def sweep_arc_thresholds(tasks, X_tr, y_tr, X_val, y_val, feat_dim, cfg, device,
                          eps_values=None, theta_values=None):
    """
    Grid search over ε (Assumption 1) and ϑ (Assumption 2).
    Base classifier trained once without ARC, then each (ε, ϑ) combo
    is evaluated on validation set with a fresh clf_arc copy.
    Matches paper Fig 5 ablation style.
    """
    if eps_values is None:
        eps_values   = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
    if theta_values is None:
        theta_values = [0.02, 0.05, 0.08, 0.10, 0.15, 0.20]

    # Train base classifier (no ARC, no replay)
    clf_base = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
    clf_checkpoints_sweep = {}
    ewc_sweep = None
    for task in tasks:
        tid = task['task_id']
        if tid > 0:
            clf_base.grow(len(task['new_classes']), device)
        clf_base = train_classifier_on_task(
            clf_base, X_tr, y_tr, X_val, y_val,
            task['all_classes'], cfg['clf_epochs'], cfg['clf_lr'], cfg['clf_batch'], device,
            es_patience=cfg.get('es_patience', 10),
            es_min_delta=cfg.get('es_min_delta', 1e-4),
            ewc=ewc_sweep, ewc_lambda=cfg.get('ewc_lambda', 400.0),
        )
        clf_checkpoints_sweep[tid] = copy.deepcopy(clf_base.state_dict())
        # Update EWC after each task
        if tid < len(tasks) - 1:
            tr_mask = np.isin(y_tr, task['all_classes'])
            ewc_ds  = MolDataset(X_tr[tr_mask], y_tr[tr_mask])
            ewc_dl  = DataLoader(ewc_ds, batch_size=cfg['clf_batch'], shuffle=False)
            ewc_sweep = EWC(clf_base, ewc_dl, device, len(task['all_classes']))

    print(f'\nSweeping ε ∈ {eps_values}  ×  θ ∈ {theta_values}')
    rows = []
    best_f1, best_eps, best_theta = -1.0, eps_values[0], theta_values[0]

    for eps in eps_values:
        for theta in theta_values:
            task_f1s = []
            for task in tasks:
                tid      = task['task_id']
                seen_cls = task['all_classes']
                if tid == 0:
                    continue

                val_mask = np.isin(y_val, seen_cls)
                X_v, y_v = X_val[val_mask], y_val[val_mask]
                if len(X_v) == 0:
                    continue

                # Fresh clf_arc per (eps, theta) — prevents contamination
                clf_arc = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
                for prev_task in tasks[:tid + 1]:
                    if prev_task['task_id'] > 0:
                        clf_arc.grow(len(prev_task['new_classes']), device)
                clf_arc.load_state_dict(clf_checkpoints_sweep[tid])
                clf_arc.eval()

                Xv = torch.tensor(X_v, dtype=torch.float32).to(device)
                with torch.no_grad():
                    logits_all  = clf_arc(Xv)
                batch_theta_val = compute_batch_theta(logits_all, tid, cfg['classes_per_task'], theta)

                val_preds = []
                for i in range(len(Xv)):
                    xi = Xv[i:i+1]
                    with torch.no_grad():
                        li = clf_arc(xi)
                    decisions, _, pred_i = out_of_task_detection(
                        li, tid, cfg['classes_per_task'], eps, theta, batch_theta=batch_theta_val)
                    decision     = decisions[0]
                    pseudo_label = pred_i[0].item()

                    if decision == 'retention':
                        adaptive_retention_step(clf_arc, xi, pseudo_label, cfg['arc_lr'], device)
                        with torch.no_grad():
                            clf_arc.eval()
                            li = clf_arc(xi)
                        pred_final = li.argmax(1).item()
                    elif decision == 'correction':
                        with torch.no_grad():
                            clf_arc.eval()
                            li = clf_arc(xi)
                        _, pred_final = compute_tss(li[0], tid, cfg['classes_per_task'], cfg['arc_temp'])
                    else:
                        pred_final = pseudo_label
                    val_preds.append(pred_final)

                present = np.unique(y_v)
                task_f1s.append(float(f1_score(y_v, val_preds, labels=present,
                                                average='macro', zero_division=0)))

            if not task_f1s:
                continue
            macro_f1 = float(np.mean(task_f1s))
            rows.append({'epsilon': eps, 'theta': theta, 'val_macro_f1': macro_f1})
            if macro_f1 > best_f1:
                best_f1, best_eps, best_theta = macro_f1, eps, theta

    sweep_df = pd.DataFrame(rows).sort_values('val_macro_f1', ascending=False)
    print('\nTop-5 threshold combinations:')
    print(sweep_df.head(5).to_string(index=False))
    print(f'\nBest: ε={best_eps}  θ={best_theta}  val_macro_f1={best_f1:.4f}')
    return best_eps, best_theta, sweep_df


print('Hyperparameter sweep defined (fresh clf_arc per combo)')

Hyperparameter sweep defined (fresh clf_arc per combo)


## Step 11 — Evaluation Functions
AUROC computed from **baseline logits** (before any ARC adaptation) — fixes AUROC drop issue.  
OTD counts (retention/correction/current) reported per paper Table 4.

In [22]:
def compute_auroc(y_t, probs_np, seen_cls, n_total_cls):
    valid_seen = [c for c in seen_cls if c < n_total_cls]
    if len(valid_seen) < 2:
        return float('nan')
    y_t_arr    = np.asarray(y_t)
    y_bin      = np.stack([(y_t_arr == c).astype(int) for c in valid_seen], axis=1)
    probs_seen = probs_np[:, valid_seen]
    probs_seen = probs_seen / (probs_seen.sum(axis=1, keepdims=True) + 1e-8)
    auroc_list = []
    for col_i in range(len(valid_seen)):
        col_true = y_bin[:, col_i]
        col_prob = probs_seen[:, col_i]
        if col_true.sum() == 0 or col_true.sum() == len(col_true):
            continue
        try:
            auroc_list.append(roc_auc_score(col_true, col_prob))
        except Exception:
            pass
    return float(np.mean(auroc_list)) if auroc_list else float('nan')


def evaluate_task(
    clf, X_test, y_test, seen_cls,
    task_id=None, n_cpt=None, epsilon=None, theta=None, temp=None,
    arc_lr=None, use_arc=False, arc_mode='full', device='cpu',
    persistent_clf_arc=None,
):
    """
    Evaluate one task slice.
    - AUROC uses baseline (pre-ARC) softmax probs — fixes AUROC corruption bug.
    - OTD counts printed per sample batch for transparency (matches paper Table 4).
    - persistent_clf_arc carries retention state across samples in a task (memory-free).
    arc_mode: 'full' | 'retention' | 'correction'  (for ablation study)
    """
    mask = np.isin(y_test, seen_cls)
    X_t, y_t = X_test[mask], y_test[mask]
    if len(X_t) == 0:
        return {'accuracy': 0.0, 'macro_f1': 0.0, 'auroc': float('nan'),
                'otd_retention': 0, 'otd_correction': 0, 'otd_current': 0}, persistent_clf_arc

    Xt = torch.tensor(X_t, dtype=torch.float32).to(device)
    clf.eval()
    with torch.no_grad():
        logits_all = clf(Xt)
    n_total_cls = logits_all.shape[1]

    # Baseline probs for AUROC — computed BEFORE any ARC adaptation
    with torch.no_grad():
        baseline_probs_np = F.softmax(logits_all, dim=-1).cpu().numpy()

    if use_arc and task_id is not None and task_id > 0:
        clf_arc = persistent_clf_arc if persistent_clf_arc is not None else copy.deepcopy(clf)

        with torch.no_grad():
            clf_arc.eval()
            logits_for_theta = clf_arc(Xt)
        batch_theta = compute_batch_theta(logits_for_theta, task_id, n_cpt, theta)

        otd_counts = {'retention': 0, 'correction': 0, 'current': 0}
        preds      = []

        for i in range(len(Xt)):
            xi = Xt[i:i+1]
            with torch.no_grad():
                clf_arc.eval()
                li_otd = clf_arc(xi)

            decisions, _, pred_i = out_of_task_detection(
                li_otd, task_id, n_cpt, epsilon, theta, batch_theta=batch_theta)
            decision     = decisions[0]
            pseudo_label = pred_i[0].item()
            otd_counts[decision] += 1

            if decision == 'retention' and arc_mode in ('full', 'retention'):
                adaptive_retention_step(clf_arc, xi, pseudo_label, arc_lr, device)
                with torch.no_grad():
                    clf_arc.eval()
                    li_final = clf_arc(xi)
                pred_final = li_final.argmax(1).item()

            elif decision == 'correction' and arc_mode in ('full', 'correction'):
                with torch.no_grad():
                    clf_arc.eval()
                    li_final = clf_arc(xi)
                _, pred_final = compute_tss(li_final[0], task_id, n_cpt, temp)

            else:
                pred_final = pseudo_label

            preds.append(pred_final)

        preds = np.array(preds)
        n = len(Xt)
        print(f'    OTD [{arc_mode}]: '
              f'retention={otd_counts["retention"]}/{n} ({100*otd_counts["retention"]/n:.1f}%)  '
              f'correction={otd_counts["correction"]}/{n} ({100*otd_counts["correction"]/n:.1f}%)  '
              f'current={otd_counts["current"]}/{n} ({100*otd_counts["current"]/n:.1f}%)  '
              f'batch_θ={batch_theta:.3f}')
    else:
        clf_arc    = persistent_clf_arc
        otd_counts = {'retention': 0, 'correction': 0, 'current': 0}
        with torch.no_grad():
            preds = logits_all.argmax(1).cpu().numpy()

    accuracy = float((preds == y_t).mean())
    present  = np.unique(y_t)
    macro_f1 = float(f1_score(y_t, preds, labels=present, average='macro', zero_division=0))
    # AUROC always uses baseline probs (not post-ARC adapted logits)
    auroc    = compute_auroc(y_t, baseline_probs_np, seen_cls, n_total_cls)

    return {
        'accuracy'       : accuracy,
        'macro_f1'       : macro_f1,
        'auroc'          : auroc,
        'otd_retention'  : otd_counts['retention'],
        'otd_correction' : otd_counts['correction'],
        'otd_current'    : otd_counts['current'],
    }, clf_arc


print('evaluate_task defined (AUROC=baseline probs | OTD counts reported)')

evaluate_task defined (AUROC=baseline probs | OTD counts reported)


## Step 12 — Full CIL Pipeline (Memory-Free)
Single pipeline for both baseline (no ARC) and ARC conditions.  
EWC updated after each task during training.

In [23]:
def run_cil_pipeline(
    tasks, X_tr, y_tr, X_val, y_val, X_te, y_te,
    feat_dim, label_map, cfg, device, use_arc=False, arc_mode='full', name='Dataset'
):
    """
    Memory-free CIL pipeline.
    - No replay buffer at all.
    - EWC regularization protects past task weights during training.
    - ARC applied at test time only (does not modify training).
    - AUROC computed from pre-ARC baseline logits.
    """
    print(f'\n{"="*60}')
    print(f'  {name}  |  ARC={use_arc}  arc_mode={arc_mode}')
    print(f'{"="*60}')

    clf             = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
    ewc_state       = None
    persistent_clf_arc = None
    clf_checkpoints = {}
    R               = defaultdict(dict)
    peak_acc        = {}
    rows            = []

    for task in tasks:
        tid      = task['task_id']
        seen_cls = task['all_classes']
        print(f'\n-- Task {tid} | new classes: {[task["class_names"][c] for c in task["new_classes"]]} --')

        if tid > 0:
            clf.grow(len(task['new_classes']), device)

        clf = train_classifier_on_task(
            clf, X_tr, y_tr, X_val, y_val,
            seen_cls, cfg['clf_epochs'], cfg['clf_lr'], cfg['clf_batch'], device,
            es_patience  = cfg.get('es_patience', 10),
            es_min_delta = cfg.get('es_min_delta', 1e-4),
            ewc          = ewc_state,
            ewc_lambda   = cfg.get('ewc_lambda', 400.0),
        )

        # Update EWC fisher after training this task
        if tid < len(tasks) - 1:
            tr_mask  = np.isin(y_tr, seen_cls)
            ewc_ds   = MolDataset(X_tr[tr_mask], y_tr[tr_mask])
            ewc_dl   = DataLoader(ewc_ds, batch_size=cfg['clf_batch'], shuffle=False)
            ewc_state = EWC(clf, ewc_dl, device, len(seen_cls))
            print(f'  [EWC] Fisher updated for task {tid}')

        clf_checkpoints[tid] = copy.deepcopy(clf.state_dict())

        if use_arc:
            persistent_clf_arc = copy.deepcopy(clf)

        # Evaluate all seen tasks at current task boundary
        for prev_task in tasks[:tid + 1]:
            prev_tid  = prev_task['task_id']
            prev_seen = prev_task['all_classes']

            fresh_arc = copy.deepcopy(clf) if use_arc else None

            res, _ = evaluate_task(
        clf, X_te, y_te, prev_seen,
        task_id            = tid,
        n_cpt              = cfg['classes_per_task'],
        epsilon            = cfg['arc_epsilon'],
        theta              = cfg['arc_theta'],
        temp               = cfg['arc_temp'],
        arc_lr             = cfg['arc_lr'],
        use_arc            = use_arc,
        arc_mode           = arc_mode,
        device             = device,
        persistent_clf_arc = fresh_arc,
    )
            R[tid][prev_tid] = res
            if prev_tid == tid:
                peak_acc[tid] = res['accuracy']

            auroc_str = f"{res['auroc']:.4f}" if not np.isnan(res['auroc']) else 'nan'
            print(f"  Eval Task {prev_tid} → acc={res['accuracy']:.4f}  f1={res['macro_f1']:.4f}  auroc={auroc_str}")

    # Aggregate final metrics
    n_tasks   = len(tasks)
    final_tid = n_tasks - 1
    acc_list, f1_list, auroc_list, forget_list = [], [], [], []
    total_ret = total_cor = total_cur = 0

    for prev_tid in range(n_tasks):
        r = R[final_tid][prev_tid]
        acc_list.append(r['accuracy'])
        f1_list.append(r['macro_f1'])
        auroc_list.append(r['auroc'])
        total_ret += r['otd_retention']
        total_cor += r['otd_correction']
        total_cur += r['otd_current']
        if prev_tid < final_tid:
            forget_list.append(peak_acc[prev_tid] - r['accuracy'])
        rows.append({
            'task_id'       : prev_tid,
            'class_names'   : str(list(tasks[prev_tid]['class_names'].values())),
            'final_acc'     : round(r['accuracy'], 4),
            'final_macro_f1': round(r['macro_f1'], 4),
            'final_auroc'   : round(r['auroc'], 4) if not np.isnan(r['auroc']) else float('nan'),
            'peak_acc'      : round(peak_acc.get(prev_tid, r['accuracy']), 4),
            'forgetting'    : round(peak_acc.get(prev_tid, r['accuracy']) - r['accuracy'], 4),
        })

    avg_acc    = float(np.mean(acc_list))
    avg_f1     = float(np.mean(f1_list))
    forgetting = float(np.mean(forget_list)) if forget_list else 0.0
    avg_auroc  = float(np.nanmean(auroc_list))

    print(f'\n{"─"*55}')
    print(f'  Average Accuracy (AB)  : {avg_acc:.4f}')
    print(f'  Average Macro-F1       : {avg_f1:.4f}')
    print(f'  Forgetting (F)         : {forgetting:.4f}')
    print(f'  Average AUROC          : {avg_auroc:.4f}')
    if use_arc:
        total = total_ret + total_cor + total_cur
        total = max(total, 1)
        print(f'  OTD totals: retention={total_ret} ({100*total_ret/total:.1f}%)  '
              f'correction={total_cor} ({100*total_cor/total:.1f}%)  '
              f'current={total_cur} ({100*total_cur/total:.1f}%)')
    print(f'{"─"*55}')

    return pd.DataFrame(rows), avg_acc, avg_f1, forgetting, avg_auroc, clf_checkpoints


print('run_cil_pipeline defined (memory-free | EWC | ARC test-time only)')

run_cil_pipeline defined (memory-free | EWC | ARC test-time only)


In [24]:
import torch
import numpy as np

def diagnose_arc_signals(tasks, X_val, y_val, ckpts, feat_dim, cfg, device, dataset_name):
    # Last task ka checkpoint use karo (sabse zyada classes wala)
    final_tid = max(ckpts.keys())

    clf = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
    for task in tasks:
        if task['task_id'] == 0:
            continue
        clf.grow(len(task['new_classes']), device)
        if task['task_id'] == final_tid:
            break
    clf.load_state_dict(ckpts[final_tid])
    clf.eval()

    all_w, all_conf = [], []

    for task in tasks[1:]:
        tid = task['task_id']
        val_mask = np.isin(y_val, task['all_classes'])
        X_v = torch.tensor(X_val[val_mask], dtype=torch.float32).to(device)
        if len(X_v) == 0:
            continue

        with torch.no_grad():
            logits = clf(X_v)

        s = cfg['classes_per_task']
        probs_all = torch.softmax(logits, dim=1).cpu().numpy()

        for i in range(len(X_v)):
            probs = probs_all[i]
            curr  = probs[tid*s : (tid+1)*s].max()
            past  = probs[:tid*s].max() if tid > 0 else 1e-9
            all_w.append(curr / (past + 1e-9))
            all_conf.append(curr)

    all_w    = np.array(all_w)
    all_conf = np.array(all_conf)

    print(f'\n=== {dataset_name} ARC Signal Diagnostics ===')
    print(f'  Confidence c  → mean={all_conf.mean():.3f}  median={np.median(all_conf):.3f}  min={all_conf.min():.3f}  max={all_conf.max():.3f}')
    print(f'  Ratio w=c/ĉ   → mean={all_w.mean():.3f}  median={np.median(all_w):.3f}  min={all_w.min():.3f}  max={all_w.max():.3f}')
    print(f'  Correction triggers (w < θ):')
    for t in [0.02, 0.05, 0.10, 0.20]:
        print(f'    θ={t:.2f} → {(all_w < t).mean()*100:.1f}%')
    print(f'  Retention eligible (c > ε):')
    for e in [0.50, 0.70, 0.80, 0.90]:
        print(f'    ε={e:.2f} → {(all_conf > e).mean()*100:.1f}%')





In [25]:
mds_res_base, mds_AB_base, mds_F1_base, mds_F_base, mds_AUROC_base, mds_ckpts_base = run_cil_pipeline(
    tasks=mds_tasks, X_tr=mds_X_tr, y_tr=mds_y_tr,
    X_val=mds_X_val, y_val=mds_y_val, X_te=mds_X_te, y_te=mds_y_te,
    feat_dim=FEAT_DIM, label_map=mds_label_map,
    cfg=CFG, device=DEVICE, use_arc=False, name='MultiDataset Baseline (no-replay)'
)


  MultiDataset Baseline (no-replay)  |  ARC=False  arc_mode=full

-- Task 0 | new classes: ['BBBP-neg', 'BBBP-pos'] --
  Epoch  10/100  loss=0.5592  val_f1=0.370  patience=7/10
  [Early Stop] epoch 13  best_val_f1=0.4571
  [EWC] Fisher updated for task 0
  Eval Task 0 → acc=0.4724  f1=0.4569  auroc=0.5402

-- Task 1 | new classes: ['ClinTox-safe', 'ClinTox-toxic'] --
  Epoch  10/100  loss=1.0142  val_f1=0.250  ewc=0.00002  patience=0/10
  Epoch  20/100  loss=0.9456  val_f1=0.231  ewc=0.00001  patience=4/10
  Epoch  30/100  loss=0.9032  val_f1=0.237  ewc=0.00002  patience=9/10
  [Early Stop] epoch 31  best_val_f1=0.2620
  [EWC] Fisher updated for task 1
  Eval Task 0 → acc=0.2931  f1=0.3896  auroc=0.5407
  Eval Task 1 → acc=0.3185  f1=0.2921  auroc=0.5605

-- Task 2 | new classes: ['HIV-inactive', 'HIV-active'] --
  Epoch  10/100  loss=1.1957  val_f1=0.144  ewc=0.00005  patience=7/10
  [Early Stop] epoch 13  best_val_f1=0.1583
  Eval Task 0 → acc=0.2241  f1=0.3347  auroc=0.5397
  Eval 

In [26]:
db_res_base, db_AB_base, db_F1_base, db_F_base, db_AUROC_base, db_ckpts_base = run_cil_pipeline(
    tasks=db_tasks, X_tr=db_X_tr, y_tr=db_y_tr,
    X_val=db_X_val, y_val=db_y_val, X_te=db_X_te, y_te=db_y_te,
    feat_dim=FEAT_DIM, label_map=db_label_map,
    cfg=CFG, device=DEVICE, use_arc=False, name='DrugBank Baseline (no-replay)'
)


  DrugBank Baseline (no-replay)  |  ARC=False  arc_mode=full

-- Task 0 | new classes: ['carbonyl', 'nitro'] --
  Epoch  10/100  loss=0.0268  val_f1=0.791  patience=6/10
  Epoch  20/100  loss=0.0106  val_f1=0.832  patience=7/10
  [Early Stop] epoch 23  best_val_f1=0.8833
  [EWC] Fisher updated for task 0
  Eval Task 0 → acc=0.9913  f1=0.6407  auroc=1.0000

-- Task 1 | new classes: ['none_detected', 'sulfonyl'] --
  Epoch  10/100  loss=0.1765  val_f1=0.809  ewc=0.00001  patience=3/10
  Epoch  20/100  loss=0.1371  val_f1=0.854  ewc=0.00001  patience=9/10
  [Early Stop] epoch 21  best_val_f1=0.8642
  Eval Task 0 → acc=0.8783  f1=0.6104  auroc=1.0000
  Eval Task 1 → acc=0.8916  f1=0.6757  auroc=0.9857

───────────────────────────────────────────────────────
  Average Accuracy (AB)  : 0.8849
  Average Macro-F1       : 0.6430
  Forgetting (F)         : 0.1130
  Average AUROC          : 0.9929
───────────────────────────────────────────────────────


In [27]:
diagnose_arc_signals(mds_tasks, mds_X_val, mds_y_val, mds_ckpts_base, FEAT_DIM, CFG, DEVICE, 'MultiDataset')
diagnose_arc_signals(db_tasks,  db_X_val,  db_y_val,  db_ckpts_base,  FEAT_DIM, CFG, DEVICE, 'DrugBank')


=== MultiDataset ARC Signal Diagnostics ===
  Confidence c  → mean=0.376  median=0.334  min=0.000  max=0.994
  Ratio w=c/ĉ   → mean=6.789  median=1.228  min=0.000  max=2640.722
  Correction triggers (w < θ):
    θ=0.02 → 2.7%
    θ=0.05 → 5.9%
    θ=0.10 → 10.5%
    θ=0.20 → 17.2%
  Retention eligible (c > ε):
    ε=0.50 → 33.2%
    ε=0.70 → 16.2%
    ε=0.80 → 8.6%
    ε=0.90 → 2.4%

=== DrugBank ARC Signal Diagnostics ===
  Confidence c  → mean=0.341  median=0.143  min=0.000  max=1.000
  Ratio w=c/ĉ   → mean=75.423  median=0.170  min=0.000  max=22306.298
  Correction triggers (w < θ):
    θ=0.02 → 28.9%
    θ=0.05 → 37.6%
    θ=0.10 → 44.4%
    θ=0.20 → 51.2%
  Retention eligible (c > ε):
    ε=0.50 → 32.2%
    ε=0.70 → 27.1%
    ε=0.80 → 22.8%
    ε=0.90 → 15.9%


## Step 13 — BWT / FWT from Checkpoints

In [28]:
def compute_bwt_fwt_from_checkpoints(
    tasks, X_te, y_te, feat_dim, clf_checkpoints, cfg, device, use_arc=False
):
    """
    BWT = 1/(T-1) * Σ (R[T][i] - R[i][i])   — negative means forgetting
    FWT = 1/(T-1) * Σ (R[i-1][i] - 1/n_classes_i)  — forward transfer
    """
    n_tasks = len(tasks)
    if n_tasks < 2:
        return float('nan'), float('nan')

    R       = {}
    clf_tmp = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)

    for task in tasks:
        tid      = task['task_id']
        seen_cls = task['all_classes']
        if tid > 0:
            clf_tmp.grow(len(task['new_classes']), device)
        clf_tmp.load_state_dict(clf_checkpoints[tid])
        clf_tmp.eval()

        R[tid] = {}
        persistent_bwt = copy.deepcopy(clf_tmp) if use_arc else None

        for prev_task in tasks[:tid + 1]:
            prev_tid  = prev_task['task_id']
            prev_seen = prev_task['all_classes']
            res, persistent_bwt = evaluate_task(
                clf_tmp, X_te, y_te, prev_seen,
                task_id=tid, n_cpt=cfg['classes_per_task'],
                epsilon=cfg['arc_epsilon'], theta=cfg['arc_theta'],
                temp=cfg['arc_temp'], arc_lr=cfg['arc_lr'],
                use_arc=use_arc, device=device,
                persistent_clf_arc=persistent_bwt,
            )
            R[tid][prev_tid] = res['accuracy']

    final_tid    = n_tasks - 1
    peak_acc_bwt = {i: R[i][i] for i in range(n_tasks)}
    bwt_vals     = [R[final_tid][i] - peak_acc_bwt[i] for i in range(n_tasks - 1)]
    bwt          = float(np.mean(bwt_vals))

    fwt_vals = []
    for i in range(1, n_tasks):
        n_cls_i  = tasks[i]['n_classes']
        chance_i = 1.0 / n_cls_i
        fwt_vals.append(R[i][i] - chance_i)
    fwt = float(np.mean(fwt_vals)) if fwt_vals else float('nan')

    return bwt, fwt


print('BWT/FWT defined')

BWT/FWT defined


## Step 14 — Ablation Study (Paper Table 4 equivalent)
Four conditions on the **same** base classifier (trained once, no ARC):
1. Baseline — no ARC
2. Retention-only — only Adaptive Retention fires
3. Correction-only — only Adaptive Correction fires  
4. Full ARC — both components active

OTD counts (how many samples hit retention vs correction vs current) reported
for each condition — matches paper Table 4 analysis.

In [29]:
def run_ablation_study(
    tasks, X_tr, y_tr, X_val, y_val, X_te, y_te,
    feat_dim, label_map, cfg, device, dataset_name='Dataset'
):
    """
    Ablation study — ARC paper Table 4 equivalent.
    Single base classifier trained once (memory-free + EWC).
    All four conditions evaluated on same weights.
    OTD counts printed per condition for analysis.
    """
    print(f'\n{"="*60}')
    print(f'  ABLATION STUDY — {dataset_name}')
    print(f'{"="*60}')

    # ── Train shared base classifier ─────────────────────────────────────────
    clf_base        = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
    clf_checkpoints = {}
    ewc_state       = None

    print('\nTraining base classifier (shared across all ablation conditions)...')
    for task in tasks:
        tid      = task['task_id']
        seen_cls = task['all_classes']
        print(f'  Task {tid}: {[task["class_names"][c] for c in task["new_classes"]]}')
        if tid > 0:
            clf_base.grow(len(task['new_classes']), device)
        clf_base = train_classifier_on_task(
            clf_base, X_tr, y_tr, X_val, y_val,
            seen_cls, cfg['clf_epochs'], cfg['clf_lr'], cfg['clf_batch'], device,
            es_patience  = cfg.get('es_patience', 10),
            es_min_delta = cfg.get('es_min_delta', 1e-4),
            ewc          = ewc_state,
            ewc_lambda   = cfg.get('ewc_lambda', 400.0),
        )
        clf_checkpoints[tid] = copy.deepcopy(clf_base.state_dict())
        if tid < len(tasks) - 1:
            tr_mask   = np.isin(y_tr, seen_cls)
            ewc_ds    = MolDataset(X_tr[tr_mask], y_tr[tr_mask])
            ewc_dl    = DataLoader(ewc_ds, batch_size=cfg['clf_batch'], shuffle=False)
            ewc_state = EWC(clf_base, ewc_dl, device, len(seen_cls))

    # ── Four ablation conditions ─────────────────────────────────────────────
    conditions = [
        ('Baseline',        False, 'full'),
        ('Retention-only',  True,  'retention'),
        ('Correction-only', True,  'correction'),
        ('Full ARC',        True,  'full'),
    ]
    ablation_rows = []
    n_tasks   = len(tasks)
    final_tid = n_tasks - 1

    # Compute peak accuracy for forgetting metric (without ARC, at each task step)
    peak_accs = {}
    for task in tasks:
        tid   = task['task_id']
        clf_t = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
        for t2 in tasks[:tid + 1]:
            if t2['task_id'] > 0:
                clf_t.grow(len(t2['new_classes']), device)
        clf_t.load_state_dict(clf_checkpoints[tid])
        clf_t.eval()
        res_peak, _ = evaluate_task(
            clf_t, X_te, y_te, task['all_classes'],
            task_id=tid, n_cpt=cfg['classes_per_task'],
            epsilon=cfg['arc_epsilon'], theta=cfg['arc_theta'],
            temp=cfg['arc_temp'], arc_lr=cfg['arc_lr'],
            use_arc=False, device=device,
        )
        peak_accs[tid] = res_peak['accuracy']

    for cond_name, use_arc, arc_mode in conditions:
        print(f'\n--- Condition: {cond_name} ---')

        clf_eval = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
        for task in tasks:
            if task['task_id'] > 0:
                clf_eval.grow(len(task['new_classes']), device)
        clf_eval.load_state_dict(clf_checkpoints[final_tid])
        clf_eval.eval()

        persistent_clf_arc = copy.deepcopy(clf_eval) if use_arc else None
        acc_list, f1_list, auroc_list, forget_list = [], [], [], []
        total_ret = total_cor = total_cur = 0

        for prev_task in tasks:
            prev_tid  = prev_task['task_id']
            prev_seen = prev_task['all_classes']

            fresh_arc_abl = copy.deepcopy(clf_eval) if use_arc else None  # ← FIXED

            res, _ = evaluate_task(
              clf_eval, X_te, y_te, prev_seen,
              task_id            = final_tid,
              n_cpt              = cfg['classes_per_task'],
              epsilon            = cfg['arc_epsilon'],
              theta              = cfg['arc_theta'],
              temp               = cfg['arc_temp'],
              arc_lr             = cfg['arc_lr'],
              use_arc            = use_arc,
              arc_mode           = arc_mode,
              device             = device,
              persistent_clf_arc = fresh_arc_abl,
           )
            acc_list.append(res['accuracy'])
            f1_list.append(res['macro_f1'])
            auroc_list.append(res['auroc'])
            total_ret += res['otd_retention']
            total_cor += res['otd_correction']
            total_cur += res['otd_current']
            if prev_tid < final_tid:
                forget_list.append(peak_accs[prev_tid] - res['accuracy'])
            auroc_str = f"{res['auroc']:.4f}" if not np.isnan(res['auroc']) else 'nan'
            print(f"  Task {prev_tid}: acc={res['accuracy']:.4f}  f1={res['macro_f1']:.4f}  "
                 f"auroc={auroc_str}")

        avg_acc    = float(np.mean(acc_list))
        avg_f1     = float(np.mean(f1_list))
        forgetting = float(np.mean(forget_list)) if forget_list else 0.0
        avg_auroc  = float(np.nanmean(auroc_list))

        total_otd = max(total_ret + total_cor + total_cur, 1)
        print(f'  → AB={avg_acc:.4f}  F1={avg_f1:.4f}  F={forgetting:.4f}  AUROC={avg_auroc:.4f}')
        if use_arc:
            print(f'  OTD: retention={total_ret}({100*total_ret/total_otd:.1f}%)  '
                  f'correction={total_cor}({100*total_cor/total_otd:.1f}%)  '
                  f'current={total_cur}({100*total_cur/total_otd:.1f}%)')

        ablation_rows.append({
            'Dataset'        : dataset_name,
            'Condition'      : cond_name,
            'AB'             : round(avg_acc,    4),
            'Macro-F1'       : round(avg_f1,     4),
            'F'              : round(forgetting, 4),
            'AUROC'          : round(avg_auroc,  4),
            'OTD_retention%' : round(100 * total_ret / total_otd, 1),
            'OTD_correction%': round(100 * total_cor / total_otd, 1),
        })

    ablation_df  = pd.DataFrame(ablation_rows)
    baseline_ab  = ablation_df.loc[ablation_df['Condition'] == 'Baseline', 'AB'].values[0]
    baseline_f1  = ablation_df.loc[ablation_df['Condition'] == 'Baseline', 'Macro-F1'].values[0]
    ablation_df['ΔAB']       = (ablation_df['AB']       - baseline_ab).round(4)
    ablation_df['ΔMacro-F1'] = (ablation_df['Macro-F1'] - baseline_f1).round(4)

    print(f'\n{"─"*60}')
    print(f'  ABLATION SUMMARY — {dataset_name}  (Table 4 equivalent)')
    print(f'{"─"*60}')
    print(ablation_df.to_string(index=False))
    return ablation_df


print('run_ablation_study defined (Baseline | Retention-only | Correction-only | Full ARC)')

run_ablation_study defined (Baseline | Retention-only | Correction-only | Full ARC)


In [30]:
print(CFG['classes_per_task'])   # classes per task
print(len(mds_tasks))            # total CIL tasks (should be 3)
# MDS: 3 tasks x 2 classes = 6 classes total

# Feature quality check — LogisticRegression baseline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Task 0 only (BBBP)
mask_tr = np.isin(mds_y_tr, mds_tasks[0]['all_classes'])
mask_va = np.isin(mds_y_val, mds_tasks[0]['all_classes'])
lr = LogisticRegression(max_iter=1000)
lr.fit(mds_X_tr[mask_tr], mds_y_tr[mask_tr])
pred = lr.predict(mds_X_val[mask_va])
print(f'MDS Task-0 (BBBP) only → F1: {f1_score(mds_y_val[mask_va], pred, average="macro"):.4f}')

# All tasks combined
lr2 = LogisticRegression(max_iter=1000)
lr2.fit(mds_X_tr, mds_y_tr)
pred2 = lr2.predict(mds_X_val)
print(f'MDS All tasks       → F1: {f1_score(mds_y_val, pred2, average="macro"):.4f}')

# DrugBank
lr3 = LogisticRegression(max_iter=1000)
lr3.fit(db_X_tr, db_y_tr)
pred3 = lr3.predict(db_X_val)
print(f'DrugBank All tasks  → F1: {f1_score(db_y_val, pred3, average="macro"):.4f}')


2
3
MDS Task-0 (BBBP) only → F1: 0.4282
MDS All tasks       → F1: 0.2099
DrugBank All tasks  → F1: 0.8924


## Run — Hyperparameter Sweep

In [31]:
print('=== Multi-Dataset Threshold Sweep ===')
mds_best_eps, mds_best_theta, mds_sweep_df = sweep_arc_thresholds(
    mds_tasks, mds_X_tr, mds_y_tr, mds_X_val, mds_y_val,
    FEAT_DIM, CFG, DEVICE,
    eps_values   = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95],
    theta_values = [0.02, 0.05, 0.08, 0.10, 0.15, 0.20],
)

print('\n=== DrugBank Threshold Sweep ===')
db_best_eps, db_best_theta, db_sweep_df = sweep_arc_thresholds(
    db_tasks, db_X_tr, db_y_tr, db_X_val, db_y_val,
    FEAT_DIM, CFG, DEVICE,
    eps_values   = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95],
    theta_values = [0.02, 0.05, 0.08, 0.10, 0.15, 0.20],
)

CFG_MDS_ARC = dict(CFG, arc_epsilon=mds_best_eps, arc_theta=mds_best_theta)
CFG_DB_ARC  = dict(CFG, arc_epsilon=db_best_eps,  arc_theta=db_best_theta)
print(f'\nFinal: MDS ε={mds_best_eps} θ={mds_best_theta} | DB ε={db_best_eps} θ={db_best_theta}')


=== Multi-Dataset Threshold Sweep ===
  Epoch  10/100  loss=0.5794  val_f1=0.377  patience=3/10
  [Early Stop] epoch 17  best_val_f1=0.5294
  Epoch  10/100  loss=0.9946  val_f1=0.247  ewc=0.00001  patience=4/10
  Epoch  20/100  loss=0.9356  val_f1=0.237  ewc=0.00003  patience=1/10
  Epoch  30/100  loss=0.9024  val_f1=0.252  ewc=0.00001  patience=4/10
  Epoch  40/100  loss=0.8745  val_f1=0.250  ewc=0.00001  patience=5/10
  [Early Stop] epoch 45  best_val_f1=0.2722
  Epoch  10/100  loss=1.1582  val_f1=0.146  ewc=0.00004  patience=2/10
  [Early Stop] epoch 18  best_val_f1=0.1698

Sweeping ε ∈ [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]  ×  θ ∈ [0.02, 0.05, 0.08, 0.1, 0.15, 0.2]

Top-5 threshold combinations:
 epsilon  theta  val_macro_f1
     0.9   0.08      0.201162
     0.9   0.02      0.201162
     0.9   0.20      0.201162
     0.9   0.15      0.201162
     0.9   0.10      0.201162

Best: ε=0.9  θ=0.02  val_macro_f1=0.2012

=== DrugBank Threshold Sweep ===
  Epoch  10/100  loss=0.0255  val_f1=0.83

## Run — Multi-Dataset BBBP+ClinTox+HIV (Baseline + ARC)


In [32]:
# Memory-free baseline (no ARC, no replay)
mds_res_base, mds_AB_base, mds_F1_base, mds_F_base, mds_AUROC_base, mds_ckpts_base = run_cil_pipeline(
    tasks=mds_tasks, X_tr=mds_X_tr, y_tr=mds_y_tr,
    X_val=mds_X_val, y_val=mds_y_val, X_te=mds_X_te, y_te=mds_y_te,
    feat_dim=FEAT_DIM, label_map=mds_label_map,
    cfg=CFG, device=DEVICE, use_arc=False, name='MultiDataset Baseline (no-replay)'
)
print('\nMulti-Dataset Baseline Results:')
print(mds_res_base.to_string(index=False))



  MultiDataset Baseline (no-replay)  |  ARC=False  arc_mode=full

-- Task 0 | new classes: ['BBBP-neg', 'BBBP-pos'] --
  Epoch  10/100  loss=0.5675  val_f1=0.453  patience=2/10
  Epoch  20/100  loss=0.5273  val_f1=0.337  patience=5/10
  [Early Stop] epoch 25  best_val_f1=0.5138
  [EWC] Fisher updated for task 0
  Eval Task 0 → acc=0.5483  f1=0.5140  auroc=0.5488

-- Task 1 | new classes: ['ClinTox-safe', 'ClinTox-toxic'] --
  Epoch  10/100  loss=0.9730  val_f1=0.216  ewc=0.00003  patience=4/10
  Epoch  20/100  loss=0.9269  val_f1=0.200  ewc=0.00002  patience=7/10
  [Early Stop] epoch 23  best_val_f1=0.2453
  [EWC] Fisher updated for task 1
  Eval Task 0 → acc=0.2172  f1=0.3141  auroc=0.5521
  Eval Task 1 → acc=0.2846  f1=0.2657  auroc=0.5558

-- Task 2 | new classes: ['HIV-inactive', 'HIV-active'] --
  Epoch  10/100  loss=1.2071  val_f1=0.145  ewc=0.00009  patience=8/10
  [Early Stop] epoch 12  best_val_f1=0.1585
  Eval Task 0 → acc=0.1586  f1=0.2552  auroc=0.5305
  Eval Task 1 → acc=

In [33]:
# ARC with tuned thresholds
print(f'MDS ARC: ε={mds_best_eps}  θ={mds_best_theta}')
mds_res_arc, mds_AB_arc, mds_F1_arc, mds_F_arc, mds_AUROC_arc, mds_ckpts_arc = run_cil_pipeline(
    tasks=mds_tasks, X_tr=mds_X_tr, y_tr=mds_y_tr,
    X_val=mds_X_val, y_val=mds_y_val, X_te=mds_X_te, y_te=mds_y_te,
    feat_dim=FEAT_DIM, label_map=mds_label_map,
    cfg=CFG_MDS_ARC, device=DEVICE, use_arc=True, name='MultiDataset + ARC'
)
print('\nMulti-Dataset + ARC Results:')
print(mds_res_arc.to_string(index=False))


MDS ARC: ε=0.9  θ=0.02

  MultiDataset + ARC  |  ARC=True  arc_mode=full

-- Task 0 | new classes: ['BBBP-neg', 'BBBP-pos'] --
  Epoch  10/100  loss=0.5619  val_f1=0.338  patience=5/10
  [Early Stop] epoch 15  best_val_f1=0.5216
  [EWC] Fisher updated for task 0
  Eval Task 0 → acc=0.5828  f1=0.5225  auroc=0.5333

-- Task 1 | new classes: ['ClinTox-safe', 'ClinTox-toxic'] --
  Epoch  10/100  loss=1.0071  val_f1=0.221  ewc=0.00003  patience=8/10
  Epoch  20/100  loss=0.9373  val_f1=0.234  ewc=0.00001  patience=9/10
  [Early Stop] epoch 21  best_val_f1=0.2651
  [EWC] Fisher updated for task 1
    OTD [full]: retention=0/290 (0.0%)  correction=31/290 (10.7%)  current=259/290 (89.3%)  batch_θ=1.200
  Eval Task 0 → acc=0.3586  f1=0.4170  auroc=0.5275
    OTD [full]: retention=0/383 (0.0%)  correction=37/383 (9.7%)  current=346/383 (90.3%)  batch_θ=1.200
  Eval Task 1 → acc=0.3551  f1=0.3108  auroc=0.5626

-- Task 2 | new classes: ['HIV-inactive', 'HIV-active'] --
  Epoch  10/100  loss=1.200

## Run — DrugBank (Baseline + ARC)

In [34]:
db_res_base, db_AB_base, db_F1_base, db_F_base, db_AUROC_base, db_ckpts_base = run_cil_pipeline(
    tasks=db_tasks, X_tr=db_X_tr, y_tr=db_y_tr,
    X_val=db_X_val, y_val=db_y_val, X_te=db_X_te, y_te=db_y_te,
    feat_dim=FEAT_DIM, label_map=db_label_map,
    cfg=CFG, device=DEVICE, use_arc=False, name='DrugBank Baseline (no-replay)'
)

print(f'\nDrugBank ARC: ε={db_best_eps}  θ={db_best_theta}')
db_res_arc, db_AB_arc, db_F1_arc, db_F_arc, db_AUROC_arc, db_ckpts_arc = run_cil_pipeline(
    tasks=db_tasks, X_tr=db_X_tr, y_tr=db_y_tr,
    X_val=db_X_val, y_val=db_y_val, X_te=db_X_te, y_te=db_y_te,
    feat_dim=FEAT_DIM, label_map=db_label_map,
    cfg=CFG_DB_ARC, device=DEVICE, use_arc=True, name='DrugBank + ARC'
)


  DrugBank Baseline (no-replay)  |  ARC=False  arc_mode=full

-- Task 0 | new classes: ['carbonyl', 'nitro'] --
  Epoch  10/100  loss=0.0277  val_f1=0.791  patience=3/10
  Epoch  20/100  loss=0.0103  val_f1=0.883  patience=6/10
  [Early Stop] epoch 24  best_val_f1=0.9158
  [EWC] Fisher updated for task 0
  Eval Task 0 → acc=0.9913  f1=0.6407  auroc=1.0000

-- Task 1 | new classes: ['none_detected', 'sulfonyl'] --
  Epoch  10/100  loss=0.1761  val_f1=0.814  ewc=0.00002  patience=3/10
  Epoch  20/100  loss=0.1317  val_f1=0.862  ewc=0.00001  patience=1/10
  [Early Stop] epoch 29  best_val_f1=0.8853
  Eval Task 0 → acc=0.8835  f1=0.6119  auroc=1.0000
  Eval Task 1 → acc=0.8979  f1=0.6763  auroc=0.9882

───────────────────────────────────────────────────────
  Average Accuracy (AB)  : 0.8907
  Average Macro-F1       : 0.6441
  Forgetting (F)         : 0.1078
  Average AUROC          : 0.9941
───────────────────────────────────────────────────────

DrugBank ARC: ε=0.6  θ=0.02

  DrugBank + 

## Compute BWT / FWT

In [35]:
mds_BWT_base, mds_FWT_base = compute_bwt_fwt_from_checkpoints(
    mds_tasks, mds_X_te, mds_y_te, FEAT_DIM, mds_ckpts_base, CFG, DEVICE, use_arc=False)
mds_BWT_arc, mds_FWT_arc = compute_bwt_fwt_from_checkpoints(
    mds_tasks, mds_X_te, mds_y_te, FEAT_DIM, mds_ckpts_arc, CFG_MDS_ARC, DEVICE, use_arc=True)

db_BWT_base, db_FWT_base = compute_bwt_fwt_from_checkpoints(
    db_tasks, db_X_te, db_y_te, FEAT_DIM, db_ckpts_base, CFG, DEVICE, use_arc=False)
db_BWT_arc, db_FWT_arc = compute_bwt_fwt_from_checkpoints(
    db_tasks, db_X_te, db_y_te, FEAT_DIM, db_ckpts_arc, CFG_DB_ARC, DEVICE, use_arc=True)

print('BWT/FWT computed')
print(f'MDS      → Baseline: BWT={mds_BWT_base:.4f} FWT={mds_FWT_base:.4f}')
print(f'MDS      → ARC:      BWT={mds_BWT_arc:.4f}  FWT={mds_FWT_arc:.4f}')
print(f'DrugBank → Baseline: BWT={db_BWT_base:.4f}  FWT={db_FWT_base:.4f}')
print(f'DrugBank → ARC:      BWT={db_BWT_arc:.4f}   FWT={db_FWT_arc:.4f}')


    OTD [full]: retention=0/290 (0.0%)  correction=31/290 (10.7%)  current=259/290 (89.3%)  batch_θ=1.200
    OTD [full]: retention=0/383 (0.0%)  correction=37/383 (9.7%)  current=346/383 (90.3%)  batch_θ=1.200
    OTD [full]: retention=0/290 (0.0%)  correction=15/290 (5.2%)  current=275/290 (94.8%)  batch_θ=1.200
    OTD [full]: retention=0/383 (0.0%)  correction=19/383 (5.0%)  current=364/383 (95.0%)  batch_θ=1.200
    OTD [full]: retention=0/6495 (0.0%)  correction=1166/6495 (18.0%)  current=5329/6495 (82.0%)  batch_θ=3.000
    OTD [full]: retention=521/575 (90.6%)  correction=4/575 (0.7%)  current=50/575 (8.7%)  batch_θ=1.200
    OTD [full]: retention=536/793 (67.6%)  correction=5/793 (0.6%)  current=252/793 (31.8%)  batch_θ=1.200
BWT/FWT computed
MDS      → Baseline: BWT=-0.2366 FWT=0.1812
MDS      → ARC:      BWT=-0.2117  FWT=0.1972
DrugBank → Baseline: BWT=-0.1078  FWT=0.6479
DrugBank → ARC:      BWT=-0.0539   FWT=0.6920


## Run — Ablation Studies (Table 4 equivalent)

In [36]:
print('Running Multi-Dataset Ablation Study...')
mds_ablation_df = run_ablation_study(
    mds_tasks, mds_X_tr, mds_y_tr, mds_X_val, mds_y_val, mds_X_te, mds_y_te,
    FEAT_DIM, mds_label_map, CFG_MDS_ARC, DEVICE, dataset_name='MultiDataset'
)

print('\nRunning DrugBank Ablation Study...')
db_ablation_df = run_ablation_study(
    db_tasks, db_X_tr, db_y_tr, db_X_val, db_y_val, db_X_te, db_y_te,
    FEAT_DIM, db_label_map, CFG_DB_ARC, DEVICE, dataset_name='DrugBank'
)

combined_ablation = pd.concat([mds_ablation_df, db_ablation_df], ignore_index=True)
print('\n=== COMBINED ABLATION TABLE (Table 4 equivalent) ===')
print(combined_ablation.to_string(index=False))


Running Multi-Dataset Ablation Study...

  ABLATION STUDY — MultiDataset

Training base classifier (shared across all ablation conditions)...
  Task 0: ['BBBP-neg', 'BBBP-pos']
  Epoch  10/100  loss=0.5624  val_f1=0.399  patience=5/10
  [Early Stop] epoch 15  best_val_f1=0.4678
  Task 1: ['ClinTox-safe', 'ClinTox-toxic']
  Epoch  10/100  loss=0.9859  val_f1=0.226  ewc=0.00001  patience=7/10
  Epoch  20/100  loss=0.9338  val_f1=0.213  ewc=0.00001  patience=7/10
  Epoch  30/100  loss=0.8919  val_f1=0.233  ewc=0.00001  patience=8/10
  [Early Stop] epoch 32  best_val_f1=0.2571
  Task 2: ['HIV-inactive', 'HIV-active']
  Epoch  10/100  loss=1.1988  val_f1=0.115  ewc=0.00005  patience=6/10
  [Early Stop] epoch 14  best_val_f1=0.1675

--- Condition: Baseline ---
  Task 0: acc=0.2103  f1=0.3129  auroc=0.5389
  Task 1: acc=0.2245  f1=0.2406  auroc=0.5499
  Task 2: acc=0.4822  f1=0.1642  auroc=0.6552
  → AB=0.3057  F1=0.2392  F=0.1970  AUROC=0.5814

--- Condition: Retention-only ---
    OTD [rete

## Final Comparison Table

In [37]:
comparison = pd.DataFrame([
    {'Dataset': 'MultiDataset', 'Method': 'Baseline (no-replay)',
     'AB': round(mds_AB_base, 4), 'Macro-F1': round(mds_F1_base, 4),
     'F' : round(mds_F_base,  4), 'AUROC':    round(mds_AUROC_base, 4),
     'BWT': round(mds_BWT_base, 4), 'FWT': round(mds_FWT_base, 4)},
    {'Dataset': 'MultiDataset', 'Method': f'+ ARC (ε={mds_best_eps} θ={mds_best_theta})',
     'AB': round(mds_AB_arc,  4), 'Macro-F1': round(mds_F1_arc,  4),
     'F' : round(mds_F_arc,   4), 'AUROC':    round(mds_AUROC_arc,  4),
     'BWT': round(mds_BWT_arc, 4), 'FWT': round(mds_FWT_arc, 4)},
    {'Dataset': 'DrugBank', 'Method': 'Baseline (no-replay)',
     'AB': round(db_AB_base,  4), 'Macro-F1': round(db_F1_base,  4),
     'F' : round(db_F_base,   4), 'AUROC':    round(db_AUROC_base, 4),
     'BWT': round(db_BWT_base, 4), 'FWT': round(db_FWT_base, 4)},
    {'Dataset': 'DrugBank', 'Method': f'+ ARC (ε={db_best_eps} θ={db_best_theta})',
     'AB': round(db_AB_arc,   4), 'Macro-F1': round(db_F1_arc,   4),
     'F' : round(db_F_arc,    4), 'AUROC':    round(db_AUROC_arc,  4),
     'BWT': round(db_BWT_arc, 4), 'FWT': round(db_FWT_arc, 4)},
])

print('=' * 95)
print('                    FINAL COMPARISON TABLE (v14 — memory-free)')
print('=' * 95)
print(comparison.to_string(index=False))
print()
print('ARC vs Baseline (apples-to-apples, both memory-free):')
print(f'  MultiDataset → ΔAB={mds_AB_arc - mds_AB_base:+.4f}  ΔF1={mds_F1_arc - mds_F1_base:+.4f}'
      f'  ΔF={mds_F_arc - mds_F_base:+.4f}  ΔAUROC={mds_AUROC_arc - mds_AUROC_base:+.4f}')
print(f'  DrugBank     → ΔAB={db_AB_arc  - db_AB_base:+.4f}  ΔF1={db_F1_arc  - db_F1_base:+.4f}'
      f'  ΔF={db_F_arc  - db_F_base:+.4f}  ΔAUROC={db_AUROC_arc  - db_AUROC_base:+.4f}')
print('=' * 95)
comparison


                    FINAL COMPARISON TABLE (v14 — memory-free)
     Dataset               Method     AB  Macro-F1      F  AUROC     BWT    FWT
MultiDataset Baseline (no-replay) 0.2847    0.2067 0.2366 0.5741 -0.2366 0.1812
MultiDataset + ARC (ε=0.9 θ=0.02) 0.3235    0.2623 0.2117 0.5883 -0.2117 0.1972
    DrugBank Baseline (no-replay) 0.8907    0.6441 0.1078 0.9941 -0.1078 0.6479
    DrugBank + ARC (ε=0.6 θ=0.02) 0.9384    0.6741 0.0539 0.9952 -0.0539 0.6920

ARC vs Baseline (apples-to-apples, both memory-free):
  MultiDataset → ΔAB=+0.0387  ΔF1=+0.0556  ΔF=-0.0249  ΔAUROC=+0.0141
  DrugBank     → ΔAB=+0.0478  ΔF1=+0.0300  ΔF=-0.0539  ΔAUROC=+0.0011


,Dataset,Method,AB,Macro-F1,F,AUROC,BWT,FWT
0,MultiDataset,Baseline (no-replay),0.2847,0.2067,0.2366,0.5741,-0.2366,0.1812
1,MultiDataset,+ ARC (ε=0.9 θ=0.02),0.3235,0.2623,0.2117,0.5883,-0.2117,0.1972
2,DrugBank,Baseline (no-replay),0.8907,0.6441,0.1078,0.9941,-0.1078,0.6479
3,DrugBank,+ ARC (ε=0.6 θ=0.02),0.9384,0.6741,0.0539,0.9952,-0.0539,0.6920


## Save All Results

In [38]:
OUT = CFG['output_dir']

mds_res_base.to_csv (os.path.join(OUT, 'mds_baseline.csv'),       index=False)
mds_res_arc.to_csv  (os.path.join(OUT, 'mds_arc.csv'),            index=False)
db_res_base.to_csv  (os.path.join(OUT, 'db_baseline.csv'),        index=False)
db_res_arc.to_csv   (os.path.join(OUT, 'db_arc.csv'),             index=False)
comparison.to_csv   (os.path.join(OUT, 'comparison_v14.csv'),     index=False)
combined_ablation.to_csv(os.path.join(OUT, 'ablation_v14.csv'),   index=False)
mds_sweep_df.to_csv (os.path.join(OUT, 'mds_sweep.csv'),          index=False)
db_sweep_df.to_csv  (os.path.join(OUT, 'db_sweep.csv'),           index=False)

meta = {
    'version': 'v14',
    'memory_free': True,
    'ewc_lambda': CFG['ewc_lambda'],
    'arc_lr': CFG['arc_lr'],
    'datasets': {
        'task0': 'BBBP     (label 0=BBBP-neg,     1=BBBP-pos)',
        'task1': 'ClinTox  (label 2=ClinTox-safe, 3=ClinTox-toxic)',
        'task2': 'HIV      (label 4=HIV-inactive, 5=HIV-active)',
    },
    'multidataset': {
        'best_eps': mds_best_eps, 'best_theta': mds_best_theta,
        'baseline': {'AB': mds_AB_base, 'F1': mds_F1_base, 'F': mds_F_base, 'AUROC': mds_AUROC_base},
        'arc'     : {'AB': mds_AB_arc,  'F1': mds_F1_arc,  'F': mds_F_arc,  'AUROC': mds_AUROC_arc},
    },
    'drugbank': {
        'best_eps': db_best_eps, 'best_theta': db_best_theta,
        'baseline': {'AB': db_AB_base, 'F1': db_F1_base, 'F': db_F_base, 'AUROC': db_AUROC_base},
        'arc'     : {'AB': db_AB_arc,  'F1': db_F1_arc,  'F': db_F_arc,  'AUROC': db_AUROC_arc},
    },
    'fixes_v14': [
        'ATC removed — replaced by BBBP+ClinTox+HIV',
        '3 datasets combined into 6-class CIL (3 tasks x 2 classes)',
        'HIV imbalance handled by weighted CE in classifier',
        'DrugBank pipeline unchanged from v13',
    ]
}
with open(os.path.join(OUT, 'metadata_v14.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('All results saved to:', OUT)


All results saved to: arc_output
